# Iron Deficiency — Exploratory Data Analysis

Target: `iron_deficiency` (binary)  
Dataset: `data/processed/nhanes_merged_adults_final.csv`

**Sections**
1. Setup & Data Loading
2. Target Distribution (class imbalance)
3. Candidate Feature Correlations (all groups)
4. Selected Features (|r| > 0.05, p < 0.05)
5. Iron-Specific Lab Features ⚠️ Leakage Risk
6. Checkup Lab Features
7. Female Reproductive Features
8. Missing Value Summary
9. Feature Summary Table

---
## 1. Setup & Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = os.path.join(os.path.abspath('..'), 'data', 'processed', 'nhanes_merged_adults_final.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns')

---
## 2. Target Distribution

In [ ]:
TARGET = 'iron_deficiency'

print('Target column present:', TARGET in df.columns)
vc = df[TARGET].value_counts(dropna=False)
print('\nValue counts (including NaN):')
print(vc)

y = df[TARGET].dropna()
prevalence = y.mean()
n_pos = int(y.sum())
n_neg = int((y == 0).sum())
ratio = n_neg / n_pos

print(f'\nUsable rows (non-NaN target): {len(y):,}')
print(f'  Positive (iron deficient): {n_pos:,}')
print(f'  Negative:                  {n_neg:,}')
print(f'  Prevalence:                {prevalence:.3f} ({prevalence*100:.1f}%)')
print(f'  Class imbalance ratio:     {ratio:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
ax = axes[0]
bars = ax.bar(['Not Iron Deficient (0)', 'Iron Deficient (1)'],
              [n_neg, n_pos], color=['#4C72B0', '#DD8452'], edgecolor='white')
for bar, val in zip(bars, [n_neg, n_pos]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', fontsize=11)
ax.set_title('Target Class Distribution', fontsize=13)
ax.set_ylabel('Count')

# Pie chart
ax2 = axes[1]
ax2.pie([n_neg, n_pos],
        labels=[f'Not Deficient\n{n_neg:,} ({100-prevalence*100:.1f}%)',
                f'Iron Deficient\n{n_pos:,} ({prevalence*100:.1f}%)'],
        colors=['#4C72B0', '#DD8452'],
        startangle=90, wedgeprops=dict(edgecolor='white'))
ax2.set_title('Class Proportions', fontsize=13)

plt.suptitle(f'Iron Deficiency: {prevalence*100:.1f}% prevalence  |  {ratio:.1f}:1 imbalance',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Candidate Feature Correlations

Point-biserial correlation (equivalent to Pearson r when one variable is binary).  
Selection threshold: **|r| > 0.05** AND **p < 0.05**.

In [ ]:
# All candidate features to test
CANDIDATE_COLS = [
    # Iron-specific labs
    'ferritin_ng_ml',
    'serum_iron_ug_dl',
    'tibc_ug_dl',
    'transferrin_saturation_pct',
    'LBXUIB_uibc_serum_ug_dl',
    'transferrin_receptor_mg_l',
    # Checkup labs
    'total_cholesterol_mg_dl',
    'hdl_cholesterol_mg_dl',
    'LBDLDL_ldl_cholesterol_friedewald_mg_dl',
    'triglycerides_mg_dl',
    'fasting_glucose_mg_dl',
    'age_years',
    'bmi',
    # Questionnaire
    'dpq040___feeling_tired_or_having_little_energy',
    'huq010___general_health_condition',
    'cdq010___shortness_of_breath_on_stairs/inclines',
    'sld012___sleep_hours___weekdays_or_workdays',
    'sld013___sleep_hours___weekends',
    'slq050___ever_told_doctor_had_trouble_sleeping?',
    'pad680___minutes_sedentary_activity',
    # Female reproductive
    'rhq031___had_regular_periods_in_past_12_months',
    'rhq060___age_at_last_menstrual_period',
    'rhq540___ever_use_female_hormones?',
]

# Gender encoding for correlation test
gender_enc = (df['gender'] == 'Female').astype(float)
gender_enc[df['gender'].isna()] = np.nan

target_series = df[TARGET].dropna()

rows = []
for col in CANDIDATE_COLS + ['gender']:
    if col == 'gender':
        feat = gender_enc
        label = 'gender (female=1)'
    else:
        if col not in df.columns:
            rows.append({'feature': col, 'r': np.nan, 'p': np.nan,
                         'n': 0, 'missing_pct': 100.0, 'note': 'NOT IN DATASET'})
            continue
        feat = df[col]
        label = col

    combined = pd.concat([feat.rename('feat'), df[TARGET]], axis=1).dropna()
    if len(combined) < 30:
        rows.append({'feature': label, 'r': np.nan, 'p': np.nan,
                     'n': len(combined), 'missing_pct': np.nan, 'note': 'too few'})
        continue

    r, p = stats.pointbiserialr(combined['feat'], combined[TARGET])
    missing_pct = feat.isna().mean() * 100
    rows.append({'feature': label, 'r': round(r, 4), 'p': round(p, 4),
                 'n': len(combined), 'missing_pct': round(missing_pct, 1), 'note': ''})

corr_df = pd.DataFrame(rows).sort_values('r', ascending=False, key=abs)
corr_df['selected'] = (corr_df['r'].abs() > 0.05) & (corr_df['p'] < 0.05)
print(corr_df[['feature', 'r', 'p', 'missing_pct', 'selected']].to_string(index=False))

In [ ]:
# Colour-coded horizontal bar chart
plot_df = corr_df.dropna(subset=['r']).sort_values('r', key=abs, ascending=True)

colors = []
for _, row in plot_df.iterrows():
    if row['selected']:
        colors.append('#DD8452' if row['r'] > 0 else '#4C72B0')
    else:
        colors.append('#cccccc')

fig, ax = plt.subplots(figsize=(9, max(6, len(plot_df) * 0.38)))
bars = ax.barh(plot_df['feature'], plot_df['r'], color=colors, edgecolor='white')
ax.axvline(0.05, color='black', linestyle='--', linewidth=0.8, label='|r|=0.05 threshold')
ax.axvline(-0.05, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Point-biserial r')
ax.set_title('Feature Correlations with Iron Deficiency\n(grey = dropped, colour = selected)', fontsize=12)

pos_patch = mpatches.Patch(color='#DD8452', label='Selected (positive r)')
neg_patch = mpatches.Patch(color='#4C72B0', label='Selected (negative r)')
drop_patch = mpatches.Patch(color='#cccccc', label='Dropped (|r|≤0.05 or p≥0.05)')
ax.legend(handles=[pos_patch, neg_patch, drop_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Selected Features

**14 features** pass the |r| > 0.05 AND p < 0.05 threshold:

| Group | Feature | r | Note |
|-------|---------|---|------|
| A (Iron labs) | ferritin_ng_ml | −0.22 | ⚠️ Leakage risk |
| A (Iron labs) | serum_iron_ug_dl | −0.30 | ⚠️ Leakage risk |
| A (Iron labs) | tibc_ug_dl | +0.44 | ⚠️ Leakage risk |
| A (Iron labs) | transferrin_saturation_pct | −0.36 | ⚠️ Leakage risk |
| A (Iron labs) | LBXUIB_uibc_serum_ug_dl | +0.51 | ⚠️ Leakage risk |
| A (Iron labs) | transferrin_receptor_mg_l | +0.52 | ⚠️ Leakage risk + 71.6% missing |
| B (Checkup) | hdl_cholesterol_mg_dl | +0.07 | |
| B (Checkup) | fasting_glucose_mg_dl | −0.08 | 57% missing (fasting subsample) |
| B (Checkup) | age_years | −0.10 | |
| B (Checkup) | gender (female=1) | +0.20 | Female at higher risk |
| B (Checkup) | triglycerides_mg_dl | −0.05 | 57% missing |
| B (Checkup) | ldl_cholesterol_mg_dl | −0.05 | 58% missing |
| C (Reproductive) | rhq031 regular periods | −0.22 | NaN for males |
| C (Reproductive) | rhq060 age last period | +0.07 | NaN for males |

**Dropped** (all questionnaire features — iron deficiency is often subclinical):
- dpq040 (tired/little energy), huq010 (general health), cdq010 (SOB on stairs)
- sld012/013 (sleep hours), slq050 (trouble sleeping), pad680 (sedentary)
- total_cholesterol (|r|<0.05), bmi (|r|<0.05), rhq540 (|r|<0.05)

> **Clinical note:** Questionnaire symptoms (fatigue, dyspnoea) did not correlate with iron deficiency in NHANES. Iron deficiency without overt anaemia is frequently subclinical — patients often do not report fatigue until haemoglobin drops below anaemia thresholds.

---
## 5. Iron-Specific Lab Features ⚠️ Leakage Risk

> **Warning:** These biomarkers are typically part of the clinical definition of iron deficiency. Including them in a production screening tool where the label is *unknown* would constitute data leakage — the model would be effectively reproducing the lab criteria rather than learning to predict from upstream signals. They are included for modelling completeness (Group A) but should be excluded in checkup-constrained deployments.

In [ ]:
IRON_SPECIFIC = [
    'ferritin_ng_ml', 'serum_iron_ug_dl', 'tibc_ug_dl',
    'transferrin_saturation_pct', 'LBXUIB_uibc_serum_ug_dl', 'transferrin_receptor_mg_l',
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(IRON_SPECIFIC):
    ax = axes[i]
    data_0 = df.loc[df[TARGET] == 0, col].dropna()
    data_1 = df.loc[df[TARGET] == 1, col].dropna()

    # Clip to 1st–99th percentile for readability
    lo = df[col].quantile(0.01)
    hi = df[col].quantile(0.99)

    ax.hist(data_0.clip(lo, hi), bins=40, alpha=0.6, color='#4C72B0',
            density=True, label='Not Deficient')
    ax.hist(data_1.clip(lo, hi), bins=40, alpha=0.6, color='#DD8452',
            density=True, label='Iron Deficient')

    r_val = corr_df.loc[corr_df['feature'] == col, 'r'].values
    r_str = f'r={r_val[0]:+.2f}' if len(r_val) else ''
    ax.set_title(f'{col}\n{r_str}  ⚠️ leakage', fontsize=9)
    ax.set_xlabel(col, fontsize=8)
    ax.legend(fontsize=8)

plt.suptitle('Group A: Iron-Specific Lab Distributions by Target', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Checkup Lab Features

In [ ]:
CHECKUP_LABS = [
    'hdl_cholesterol_mg_dl',
    'fasting_glucose_mg_dl',
    'age_years',
    'triglycerides_mg_dl',
    'LBDLDL_ldl_cholesterol_friedewald_mg_dl',
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(CHECKUP_LABS):
    ax = axes[i]
    data_0 = df.loc[df[TARGET] == 0, col].dropna()
    data_1 = df.loc[df[TARGET] == 1, col].dropna()
    lo = df[col].quantile(0.01)
    hi = df[col].quantile(0.99)
    ax.hist(data_0.clip(lo, hi), bins=40, alpha=0.6, color='#4C72B0',
            density=True, label='Not Deficient')
    ax.hist(data_1.clip(lo, hi), bins=40, alpha=0.6, color='#DD8452',
            density=True, label='Iron Deficient')
    r_val = corr_df.loc[corr_df['feature'] == col, 'r'].values
    r_str = f'r={r_val[0]:+.2f}' if len(r_val) else ''
    ax.set_title(f'{col}\n{r_str}', fontsize=9)
    ax.legend(fontsize=8)

# Gender bar chart in last panel
ax = axes[5]
gender_cross = pd.crosstab(df['gender'], df[TARGET], normalize='index') * 100
gender_cross[1].plot(kind='bar', ax=ax, color='#DD8452', edgecolor='white')
ax.set_title('gender\n% Iron Deficient by Sex  r=+0.20', fontsize=9)
ax.set_xlabel('')
ax.set_ylabel('% Iron Deficient')
ax.tick_params(axis='x', rotation=0)

plt.suptitle('Group B: Checkup Lab Distributions by Target', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Female Reproductive Features

These features are NaN for males. The pipeline median-imputes them (imputed to 0 for both rhq031 and rhq060 in the male majority, which is clinically appropriate — males don't menstruate).

In [ ]:
females = df[df['gender'] == 'Female'].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# rhq031 — regular periods in past 12 months
ax = axes[0]
col = 'rhq031___had_regular_periods_in_past_12_months'
cross = pd.crosstab(females[col], females[TARGET]).rename(columns={0: 'Not Deficient', 1: 'Iron Deficient'})
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross_pct.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='white')
ax.set_title('rhq031: Regular Periods Past 12mo\n(females only, r=−0.22)', fontsize=10)
ax.set_xlabel('1=Yes, 2=No')
ax.set_ylabel('% of group')
ax.tick_params(axis='x', rotation=0)

# rhq060 — age at last menstrual period
ax2 = axes[1]
col2 = 'rhq060___age_at_last_menstrual_period'
d0 = females.loc[females[TARGET] == 0, col2].dropna()
d1 = females.loc[females[TARGET] == 1, col2].dropna()
ax2.hist(d0, bins=30, alpha=0.6, color='#4C72B0', density=True, label='Not Deficient')
ax2.hist(d1, bins=30, alpha=0.6, color='#DD8452', density=True, label='Iron Deficient')
ax2.set_title('rhq060: Age at Last Menstrual Period\n(females only, r=+0.07)', fontsize=10)
ax2.set_xlabel('Age (years)')
ax2.legend(fontsize=9)

plt.suptitle('Group C: Female Reproductive Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Missing Value Summary

In [ ]:
ALL_FEATURES = IRON_SPECIFIC + CHECKUP_LABS + ['gender',
    'rhq031___had_regular_periods_in_past_12_months',
    'rhq060___age_at_last_menstrual_period',
]

missing = (df[ALL_FEATURES].isnull().mean() * 100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_m = ['#DD8452' if v > 50 else '#4C72B0' if v < 20 else '#8da0cb'
            for v in missing.values]
bars = ax.barh(missing.index, missing.values, color=colors_m, edgecolor='white')
ax.axvline(20, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
for bar, val in zip(bars, missing.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Feature\n(red line = 50%, grey = 20%)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Feature Summary Table

In [ ]:
summary_rows = []
for _, row in corr_df.dropna(subset=['r']).iterrows():
    feat = row['feature']
    # Determine group
    if feat in IRON_SPECIFIC:
        group = 'A — Iron-Specific Labs ⚠️'
    elif feat in CHECKUP_LABS or feat == 'gender (female=1)':
        group = 'B — Checkup Labs'
    elif 'rhq031' in feat or 'rhq060' in feat or 'rhq540' in feat:
        group = 'C — Female Reproductive'
    else:
        group = 'Questionnaire (dropped)'

    summary_rows.append({
        'Group': group,
        'Feature': feat,
        'r': row['r'],
        'p': row['p'],
        'Missing %': row['missing_pct'],
        'Selected': 'YES' if row['selected'] else 'no',
    })

summary = pd.DataFrame(summary_rows).sort_values(['Selected', 'r'], ascending=[False, True], key=lambda x: x.abs() if x.name == 'r' else x)
print(summary.to_string(index=False))
print(f"\nTotal selected: {(summary['Selected']=='YES').sum()} features")

---
## Summary

### Target
- **6.05% prevalence** (450 / 7,437 rows)
- **15.5:1 class imbalance** — severe; requires `class_weight='balanced'` (LR) and `scale_pos_weight=15` (XGBoost)

### Feature Selection Result: 14 features

**Group A — Iron-Specific Labs (6 features) ⚠️ Leakage risk**  
Strong correlations (r = 0.22–0.52) but these biomarkers likely *define* the target label. Exclude from checkup-constrained deployment.

**Group B — Checkup Labs (6 features) ✅ Safe**  
Weaker but valid signal (r = 0.05–0.20). Gender (female) is the strongest single safe predictor (r=+0.20), reflecting menstrual blood loss risk.

**Group C — Female Reproductive (2 features) ✅ Safe**  
rhq031 (regular periods past 12mo) has r=−0.22 — menstruating women without regular periods are at higher risk.

**Questionnaire features: all dropped**  
Clinically coherent — iron deficiency without anaemia is often subclinical; fatigue/dyspnoea symptoms only emerge at haemoglobin levels consistent with overt anaemia.